# Sunglint basics

This tutorial computes the polarized sunglint with `coxmunk` for a given
Sun/sensor geometry and wind, then explores the effect of the wave slope
statistics, of the wind direction and of the shadowing correction.
The equations are given in [Algorithm Description](../algorithms.rst).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from coxmunk import sunglint
from coxmunk.visu import plot

## Single geometry

`sunglint(sza, vza, azi)` sets the geometry (in degrees, relative azimuth
`azi=180` when Sun and sensor are in opposition), and the `sunglint` method
returns the Stokes vector $[I, Q, U, V]$ of the glint in reflectance units for
a wind speed `ws` (m/s) and a downwind direction `wazi` (degrees).

In [ ]:
geom = sunglint(sza=30, vza=30, azi=180)
I, Q, U, V = geom.sunglint(ws=5, wazi=0, stats='bh2006', shadow=True)
print(f'I = {I:.4f}, Q = {Q:.4f}, U = {U:.2e}, V = {V:.1e}')
print(f'degree of linear polarization = {np.hypot(Q, U) / I:.3f}')

## Polar maps

A helper computes the glint over all viewing directions for a given Sun
position and wind.

In [ ]:
vza = np.linspace(0, 80, 41)
azi = np.linspace(0, 360, 91)


def glint_map(sza, ws, wazi=0, stats='bh2006', shadow=True):
    '''Stokes vector of the glint on a (azi, vza) grid, shape (4, Nazi, Nvza).'''
    data = np.empty((4, len(azi), len(vza)))
    for j, a in enumerate(azi):
        for i, v in enumerate(vza):
            data[:, j, i] = sunglint(sza, v, a).sunglint(ws, wazi, stats=stats, shadow=shadow)
    return data

In [ ]:
I, Q, U, V = glint_map(sza=30, ws=5)
dolp = np.hypot(Q, U) / I

fig, axs = plt.subplots(1, 3, subplot_kw=dict(projection='polar'), figsize=(18, 6))
p = plot()
p.add_polplot(axs[0], vza, azi, I.T, title='I', cmap=plt.cm.Spectral_r)
p.add_polplot(axs[1], vza, azi, Q.T, title='Q', cmap=plt.cm.RdBu_r, cmap_sym=True)
p.add_polplot(axs[2], vza, azi, dolp.T, title='DoLP', cmap=plt.cm.viridis, colfmt='%0.2f')
plt.tight_layout()

The glint is centred on the specular direction ($\theta_v = \theta_s$,
$\phi = 180°$), and its polarization increases toward the Brewster angle.

## Wave slope statistics

Along the principal plane ($\phi = 180°$), the three slope statistics give
different glint widths and peak values, especially at high wind speed.

In [ ]:
vza_pp = np.linspace(0, 80, 161)
fig, axs = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, ws in zip(axs, (2, 7, 14)):
    for stats in ('cm_iso', 'cm_dir', 'bh2006'):
        I_pp = [sunglint(30, v, 180).sunglint(ws, 0, stats=stats, shadow=False)[0] for v in vza_pp]
        ax.semilogy(vza_pp, I_pp, label=stats)
    ax.set_title(f'U = {ws} m/s')
    ax.set_xlabel('viewing zenith angle (deg)')
axs[0].set_ylabel('I (reflectance)')
axs[0].set_ylim(1e-4, None)
axs[0].legend()
plt.tight_layout()

## Wind direction

With anisotropic statistics, the glint pattern is elongated along the wind
direction: upwind slopes have a larger variance than crosswind slopes.

In [ ]:
fig, axs = plt.subplots(1, 2, subplot_kw=dict(projection='polar'), figsize=(12, 6))
for ax, wazi in zip(axs, (0, 90)):
    I = glint_map(sza=30, ws=10, wazi=wazi)[0]
    plot().add_polplot(ax, vza, azi, I.T, title=f'I, wind azimuth {wazi}°',
                       cmap=plt.cm.Spectral_r)
plt.tight_layout()

## Shadowing

Shadowing and hiding of the facets by the waves only matter at grazing angles.
The ratio of the glint with and without the correction is shown along the
principal plane for a low Sun.

In [ ]:
vza_pp = np.linspace(1, 89, 177)
fig, ax = plt.subplots(figsize=(8, 5))
for ws in (2, 7, 14):
    with_sh = np.array([sunglint(75, v, 180).sunglint(ws, 90, stats='bh2006', shadow=True)[0] for v in vza_pp])
    no_sh = np.array([sunglint(75, v, 180).sunglint(ws, 90, stats='bh2006', shadow=False)[0] for v in vza_pp])
    ax.plot(vza_pp, with_sh / no_sh, label=f'U = {ws} m/s')
ax.set_xlabel('viewing zenith angle (deg)')
ax.set_ylabel('I with / without shadowing')
ax.set_title('Sun zenith angle 75°, crosswind')
ax.legend()
plt.tight_layout()